In [ ]:
# Script source: GitHub repo
import os
import shutil

github_url_path = "https://github.com/abdoljh/Lamahat/tree/main/_Phase3"

# Extract the base GitHub repository URL and the subfolder path
def parse_github_path(url):
    parts = url.split('/tree/main/')
    repo_url = parts[0]
    subfolder_path = parts[1] if len(parts) > 1 else ''
    # Add .git for cloning
    repo_url_for_clone = repo_url + '.git'
    return repo_url_for_clone, subfolder_path

repo_url_for_clone, subfolder_path_in_repo = parse_github_path(github_url_path)

repo_name = repo_url_for_clone.split('/')[-1].replace('.git', '')
temp_clone_dir = os.path.join('/tmp', repo_name)
dest_dir_colab = '/content'

print(f"Cloning repository: {repo_url_for_clone}")
print(f"Target subfolder in repo: {subfolder_path_in_repo}")

# Clean up any previous clone to avoid issues
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir)
    print(f"Removed existing temporary directory: {temp_clone_dir}")

# Clone the repository
clone_command = f"git clone {repo_url_for_clone} {temp_clone_dir}"
print(f"Executing: {clone_command}")
os.system(clone_command)

# Check if cloning was successful
if not os.path.exists(temp_clone_dir):
    print(f"Error: Failed to clone repository {repo_url_for_clone}")
else:
    source_dir_to_copy = os.path.join(temp_clone_dir, subfolder_path_in_repo)
    if not os.path.exists(source_dir_to_copy):
        print(f"Error: Subfolder '{subfolder_path_in_repo}' not found in cloned repository at '{source_dir_to_copy}'")
    else:
        print(f"Source directory to copy: {source_dir_to_copy}")
        print(f"Copying contents of '{source_dir_to_copy}' directly into '{dest_dir_colab}'.")

        # Define directories to skip
        dirs_to_skip = ['artifacts', 'review']

        try:
            for item in os.listdir(source_dir_to_copy):
                if item in dirs_to_skip:
                    print(f"Skipping directory '{item}' as requested.")
                    continue

                source_item_path = os.path.join(source_dir_to_copy, item)
                dest_item_path = os.path.join(dest_dir_colab, item)

                # If item already exists in destination, remove it to avoid errors
                if os.path.exists(dest_item_path):
                    if os.path.isdir(dest_item_path):
                        shutil.rmtree(dest_item_path)
                        print(f"Removed existing directory '{dest_item_path}'.")
                    else:
                        os.remove(dest_item_path)
                        print(f"Removed existing file '{dest_item_path}'.")

                if os.path.isdir(source_item_path):
                    shutil.copytree(source_item_path, dest_item_path)
                else:
                    shutil.copy2(source_item_path, dest_item_path)
                print(f"Copied '{source_item_path}' to '{dest_item_path}'.")
            print(f"Successfully copied contents of '{source_dir_to_copy}' to '{dest_dir_colab}'.")

        except Exception as e:
            print(f"An error occurred during directory contents copy: {e}")

# Clean up the cloned repository
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir)
    print(f"Cleaned up temporary clone directory: {temp_clone_dir}")

In [ ]:
# Verify font discovery (OPTIONAL)
# !python verify_font_discovery.py


In [ ]:
# Check font paths (OPTIONAL)
# !python -c "from phase3.typography import FONT_PATHS; print(FONT_PATHS)"
# print("✅ Pre-testingvthe discovery of Amiri fonts completed!")

In [ ]:
!pip install anthropic
print("✴️ Anthropic installed!")

In [ ]:
# !pip install whisperx openai-whisper
# print("✅ whisperx and openai-whisper installed!")

In [ ]:
# !pip install arabic-reshaper python-bidi
# print("✅ Arabic_reshaper and python-bidi installed!")

In [ ]:
print("Retrieving the Anthropic & Pexels API keys ...")
from google.colab import userdata
import os

# Retrieve the Anthropic API key from Colab Secrets
anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
pexels_api_key = userdata.get('PEXELS_API_KEY')

# Set it as an environment variable for phase3_run.py to use
if anthropic_api_key:
    os.environ['ANTHROPIC_API_KEY'] = anthropic_api_key
    print("🔑 Anthropic API key loaded from secrets and set as environment variable.")
else:
    print("❌ Warning: ANTHROPIC_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

if pexels_api_key:
    os.environ['PEXELS_API_KEY'] = pexels_api_key
    print("🔑 Pexels API key loaded from secrets and set as environment variable.")
else:
    print("❌ Warning: PEXELS_API_KEY not found in Colab Secrets. Please ensure it's set correctly.")

In [ ]:
# Confirm alignment works (interpolation backend — no install needed)
!python phase3_run.py \
    --script samples/al_askari_script.txt \
    --audio output/al_askari_audio.mp3 \
    --align-only \
    --align-backend interpolated

print("✅ Confirming alignment completed!")

In [ ]:
# Step 1: plan as before — produces output/al_askari_plan_v2.json
# Regenerate the plan with the fixes
!python phase3_run.py \
    --script  samples/al_askari_script.txt \
    --audio   output/al_askari_audio.mp3 \
    --book-title "مذكرات جعفر العسكري" \
    --character-name "Jafar al-Askari" \
    --plan-only \
    --save-plan output/al_askari_plan_v2.json

print("✅ Regenerating the plan completed!")

In [ ]:
# Audit it
!python audit_plan.py output/al_askari_plan_v2.json
# Expect: ~43 shots, <10% auto-split

print("✅ Audit completed!")

In [ ]:
import os

# Step 2 (NEW): prebuild — run every source, score every candidate,
# write a review dossier.  No video render.
!python prebuild_assets.py \
    --plan          output/al_askari_plan_v2.json \
    --script        samples/al_askari_script.txt \
    --book-title    "مذكرات جعفر العسكري" \
    --character-name "Jafar al-Askari" \
    --anthropic-key "$ANTHROPIC_API_KEY" \
    --pexels-key    "$PEXELS_API_KEY" \
    --review-dir    output/review/ \
    --character-portrait my_jafar.jpg \
    --book-cover my_book_cover.jpg

In [ ]:
import os
import zipfile

output_dir = 'output/review'
zip_filename = 'review.zip'

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    files_added = []
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            # Add file to zip, preserving directory structure relative to 'output_dir'
            # Skip .mp3 files if any exist (though unlikely in review folder)
            if not file_path.endswith('.mp3'):
                zipf.write(file_path, os.path.relpath(file_path, output_dir))
                files_added.append(file_path)

print(f"🤐 Successfully created '{zip_filename}' containing: ")
if files_added:
    for f in files_added:
        print(f"  - {f}")
else:
    print("  (No files found to zip in the 'output/review' directory.)")

In [ ]:
# Standalone test runner
!python verify_user_marked.py

In [ ]:
!python verify_title_card.py --book-cover my_book.jpg

In [ ]:
!python diagnose_issue4.py --review-dir output/review/   # is everything in place?

In [ ]:
# Step 3: user reviews output/review/ — opens decisions.json, swaps a
# few `chosen_url` values, drops shot_03.jpg / shot_38.jpg overrides.

# Step 4: render — render_plan.py reads the decisions file
# The --review-dir argument consumes user's choices
!python render_plan.py \
    --plan         output/al_askari_plan_v2.json \
    --audio        output/al_askari_audio.mp3 \
    --review-dir   output/review/ \
    --output       output/final_cut.mp4 \
    --disabled--no-captions \
    > output/render.log 2>&1 &

In [ ]:
# Monitor rendering progress
import time
from IPython.display import clear_output

log_path = "output/render.log"

print("Monitoring rendering progress...")
while True:
    try:
        # Read the log file contents
        try:
            with open(log_path, "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = ""

        # Clear cell output and show the last 20 lines
        clear_output(wait=True)
        lines = log_content.splitlines()
        print("\n".join(lines[-20:]))

        # Check if the script's success signature is in the log
        if "Done in" in log_content or "Rendered video →" in log_content:
            print("\n✅ Rendering process completed successfully! Stopped monitoring.")
            break

        time.sleep(5)

    except KeyboardInterrupt:
        print("\n⚠️ Monitoring stopped manually. The script may still be running.")
        break


In [ ]:
!python diagnose_captions.py --plan output/al_askari_plan_v2.json   # inspect actual ASS events

In [ ]:
# Zip output files for exporting
import os
import zipfile

output_dir = 'output'
zip_filename = 'output_files.zip'

# Get all files in the output directory
files_to_zip = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]

# Filter out the .mp3 file
filtered_files = [f for f in files_to_zip if not f.endswith('.mp3')]

# Create the zip archive
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in filtered_files:
        # Add file to zip, preserving directory structure relative to 'output_dir'
        zipf.write(file_path, os.path.relpath(file_path, output_dir))

print(f"🤐 Successfully created '{zip_filename}' containing: ")
for f in filtered_files:
    print(f"  - {f}")

In [ ]:
# Save zipped file to Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Define the destination directory and file path
dest_dir = '/content/drive/MyDrive/_Phase3'
dest_file_path = os.path.join(dest_dir, 'output_files.zip')

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)
print(f"Destination directory '{dest_dir}' ensured to exist.")

# --- Test write access to the directory ---
test_file = os.path.join(dest_dir, 'test_write.txt')
try:
    with open(test_file, 'w') as f:
        f.write('This is a test file.\n')
    print(f"✅ Successfully wrote test file to '{test_file}'.")
    os.remove(test_file) # Clean up the test file
    print(f"Test file '{test_file}' removed.")
except Exception as e:
    print(f"Error writing test file to '{test_file}': {e}")
    print("⚠️ It seems there might be a permissions or access issue with Google Drive.")
    # Exit or raise an error if write access fails
    raise
# ----------------------------------------

shutil.copy('/content/output_files.zip', dest_file_path)
print(f"📽️ rough_cut.mp4 saved to Google Drive at '{dest_file_path}'.")